

# Encoding Language into Numbers



#### Using ASCII Code to Encode Language Into Numbers

Let's encode the text ***silent*** into numbers. We will do this encoding *character wise*.

In [ ]:
word = "silent"

In [ ]:
list_to_hold_encoded_values = []
for char in word:
  encoded_char = char.encode('ascii')
  encoded_char_as_int = int.from_bytes(encoded_char)
  list_to_hold_encoded_values.append(encoded_char_as_int)
print(list_to_hold_encoded_values)

##### There is an alternative easier way to perform the same using list comprehension in Python, shown below:

In [ ]:
encoded_values = [char for char in word.encode('ascii')]
print(encoded_values)

####  This process is called Tokenization!

**Tokenization** is the process of breaking down text into smaller units (tokens) and converting each unique token into a numerical identifier, creating a structured representation that machine learning algorithms can process mathematically.

**TensorFlow Keras** contains a library called `preprocessing` that provides a number of extremely useful tools to prepare data for machine learning.

Let us use that now.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = ['Today is a sunny day','Today is a rainy day']

tokenizer = Tokenizer(num_words = 100) # num_words=100 limits vocabulary to top 100 most frequent words

tokenizer.fit_on_texts(sentences) # Analyzes the sentences to build the vocabulary

word_index = tokenizer.word_index #Retrieves the dictionary mapping each word to its assigned integer index. Words are ranked by frequency (most frequent = index 1).

print(word_index)

##### NOTE: `Index 0` is reserved for padding.

Let’s expand the corpus with another sentence containing the word “**today**” but with a question mark after it, the results show that it would be smart enough to filter out “**today?**” as just “**today**”:

In [ ]:
sentences = ['Today is a sunny day','Today is a rainy day', 'Is it windy today?']

In [ ]:
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

In [ ]:
print(word_index)

This behavior is controlled by the filters parameter to the tokenizer, which defaults to removing all punctuation except the apostrophe character.

## Turning Sentences into Sequences

Now that you’ve seen how to take words and tokenize them into numbers, the next step is to encode the sentences into sequences of numbers.  

The tokenizer has a method for this called `text_to_sequence` — all you have to do is pass it your list of sentences, and it will give you back a list of sequences.  

In [ ]:
sequences = tokenizer.texts_to_sequences(sentences)
print(sequences)

## Any Possible Issues?

In the case of NLP, you might have many thousands of words in your training data, used in many different contexts, but you can’t have every possible word in every possible context.

So when you show your neural network some new, previously unseen text, containing previously unseen words, what might happen?  

Reduce `num_words` parameter in the above code, and rerun the steps to see what happens to the `sequences` generated?

## Out of Vocabulary Tokens (OOV token)

The OOV (Out-Of-Vocabulary) token handles words that the tokenizer hasn't seen during training or that fall outside the `num_words` limit.

In [ ]:
test_data = ['Today is a snowy day', 'Will it still rain tomorrow?']

In [ ]:
test_sequences = tokenizer.texts_to_sequences(test_data)
print(word_index)
print("\n")
print(test_sequences)

It reads sth like: `Today is a day, it rainy`

As you can see, you’ve pretty much lost all context and meaning.  

## Using OOV

In [ ]:
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index=tokenizer.word_index
print(word_index)

In [ ]:
test_sequences = tokenizer.texts_to_sequences(test_data)
print(test_sequences)

Your tokens list has a new item, `<OOV>`, and your test sentences maintain their length. Reverse-encoding them will now give `today is a <OOV> day` and `<OOV> it <OOV> rainy <OOV>`

The former is much closer to the original meaning. The latter, because most of its words aren’t in the corpus, still lacks a lot of context, but it’s a step in the right direction.  

# Padding

A neural network needs all your data to be in the same shape.

Padding is to get them to be the same size and shape. Let’s try an example:

In [ ]:
sentences = [
    'Today is a sunny day',
    'Today is a rainy day',
    'Is it sunny today?',
    'I really enjoyed walking in the snow today'
]

In [ ]:
sequences = tokenizer.texts_to_sequences(sentences)
print(sequences)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded = pad_sequences(sequences)
print(padded)

* The required number of zeros were added at the beginning.
* This is called `prepadding` (Default behavior)
* You can change the default behavior using the `padding` parameter.

In [ ]:
padded = pad_sequences(sequences, padding='post')
print(padded)

### Other functions of `pad_sequences` API

`maxlen` (To specify the desired maximum length)

In [ ]:
padded = pad_sequences(sequences, padding='post', maxlen=11)
print(padded)

`truncating` (Truncated at the end instead of the beginning)

In [ ]:
padded = pad_sequences(sequences, padding='post', maxlen=5, truncating='post')
print(padded)

The next default behavior you may have observed is that the sentences were all made to be the same length as the longest one. It’s a sensible default because it means you don’t lose any data. The trade-off is you get a lot of padding. But what if you don’t want this, perhaps because you have one crazy long sentence that means you’d have too much padding in the padded sequences?  